In [1]:
import torch
from jaxtyping import Float

In [28]:
def _make_attn_inputs(device=None):
    torch.random.manual_seed(0)
    n_queries = 32
    n_keys = 32
    batch_size = 4
    D = 64
    Q = torch.randn(batch_size, n_queries, D, device=device, requires_grad=True)
    K = torch.randn(batch_size, n_keys, D, device=device, requires_grad=True)
    V = torch.randn(batch_size, n_keys, D, device=device, requires_grad=True)
    O = torch.zeros(batch_size, n_queries, D, device=device)
    L = torch.zeros(batch_size, n_queries, device=Q.device)

    return Q, K, V, O, L

In [29]:
Q, K, V, O, L = _make_attn_inputs()


In [33]:
def flash_attn_fwd(
    Q: Float[torch.Tensor, "b queries d_q"],
    K: Float[torch.Tensor, "b keys d_q"],
    V: Float[torch.Tensor, "b keys d_v"],
    O: Float[torch.Tensor, "b queries d_v"],
    L: Float[torch.Tensor, "b queries"],
    N_QUERIES: int,
    N_KEYS: int,
    Q_TILE_SIZE: int,
    K_TILE_SIZE: int,
    scale: float,
):
    Q_TILES = (N_QUERIES + Q_TILE_SIZE - 1) // Q_TILE_SIZE
    K_TILES = (N_KEYS + K_TILE_SIZE - 1) // K_TILE_SIZE

    B = Q.size(0)

    for i in range(Q_TILES):
        Q_tile = Q[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :]  # [B_q, d]
        O_tile = O[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :]  # [B_q, d]

        prev_m = torch.zeros((B, Q_TILE_SIZE), device=Q.device)
        l = torch.zeros((B, Q_TILE_SIZE), device=Q.device)

        for j in range(K_TILES):
            # compute P
            K_tile = K[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]  # [B_k, d]

            S_tile = Q_tile @ K_tile.transpose(-1, -2) / scale  # [B_q, B_k]

            tile_m = S_tile.max(dim=-1).values
            m = torch.maximum(prev_m, tile_m)

            P_tile = torch.exp(S_tile - m.unsqueeze(-1))

            exp_correction = torch.exp(prev_m - m)
            prev_m = m

            l = exp_correction * l + P_tile.sum(dim=-1)

            # compute O
            V_tile = V[:, K_TILE_SIZE * j : K_TILE_SIZE * (j + 1), :]
            O_tile =  O_tile * exp_correction.unsqueeze(-1) + P_tile @ V_tile

        O_tile = (1 / l.unsqueeze(-1)) * O_tile

        # write results
        O[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1), :] = O_tile
        L[:, Q_TILE_SIZE * i : Q_TILE_SIZE * (i + 1)] = prev_m + torch.log(l)


flash_attn_fwd(Q, K, V, O, L, Q.size(-2), K.size(-2), 16, 16, 0.141)
O

tensor([[[-0.0550,  0.6839, -0.4356,  ...,  0.7648, -0.5318,  0.6478],
         [-0.7279,  0.8455, -0.5387,  ...,  1.3482, -2.4633,  0.5497],
         [ 0.9609,  1.5425, -1.2702,  ..., -0.7435, -0.4808,  0.6858],
         ...,
         [-1.4047,  0.0655,  0.6313,  ...,  0.7629,  0.1251,  1.1852],
         [ 0.8341,  0.5064, -0.4058,  ...,  1.1066,  0.4631,  0.8227],
         [-1.8220, -0.3451, -1.0282,  ...,  0.4132,  1.0023,  0.6766]],

        [[-0.6236, -0.4240, -0.1468,  ...,  2.1584, -0.6381,  0.2890],
         [-1.2803,  0.5295, -0.4283,  ...,  0.6370, -1.3763, -1.2876],
         [ 1.0060,  1.4529,  0.0178,  ..., -0.2465,  1.5939, -2.4683],
         ...,
         [-0.6236, -0.4240, -0.1468,  ...,  2.1584, -0.6381,  0.2890],
         [-0.6227, -0.4236, -0.1457,  ...,  2.1550, -0.6360,  0.2896],
         [ 0.0094,  0.4585, -0.9952,  ...,  0.6080,  0.0555,  0.5126]],

        [[-1.6179, -2.2276, -0.1260,  ..., -1.2309,  0.2209,  0.2490],
         [-0.4994,  0.6266,  0.5256,  ...,  1

In [32]:
def reference_attn(Q, K, V, scale):
    return torch.softmax(Q @ K.transpose(-1, -2) / scale, dim=-1) @ V

reference_attn(Q, K, V, 0.141)

tensor([[[-0.0550,  0.6839, -0.4356,  ...,  0.7648, -0.5318,  0.6478],
         [-0.7279,  0.8455, -0.5387,  ...,  1.3482, -2.4633,  0.5497],
         [ 0.9609,  1.5425, -1.2702,  ..., -0.7435, -0.4808,  0.6858],
         ...,
         [-1.4047,  0.0655,  0.6313,  ...,  0.7629,  0.1251,  1.1852],
         [ 0.8341,  0.5064, -0.4058,  ...,  1.1066,  0.4631,  0.8227],
         [-1.8220, -0.3451, -1.0282,  ...,  0.4132,  1.0023,  0.6766]],

        [[-0.6236, -0.4240, -0.1468,  ...,  2.1584, -0.6381,  0.2890],
         [-1.2803,  0.5295, -0.4283,  ...,  0.6370, -1.3763, -1.2876],
         [ 1.0060,  1.4529,  0.0178,  ..., -0.2465,  1.5939, -2.4683],
         ...,
         [-0.6236, -0.4240, -0.1468,  ...,  2.1584, -0.6381,  0.2890],
         [-0.6227, -0.4236, -0.1457,  ...,  2.1550, -0.6360,  0.2896],
         [ 0.0094,  0.4585, -0.9952,  ...,  0.6080,  0.0555,  0.5126]],

        [[-1.6179, -2.2276, -0.1260,  ..., -1.2309,  0.2209,  0.2490],
         [-0.4994,  0.6266,  0.5256,  ...,  1